In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
import warnings

warnings.filterwarnings("ignore")

In [2]:
df=pd.read_csv("data/household_power_consumption.csv",sep=";")
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [3]:
print("Null value: \n",df.isnull().sum())
print("Duplicated Values: ",df.duplicated().sum())

Null value: 
 Date                         0
Time                         0
Global_active_power          0
Global_reactive_power        0
Voltage                      0
Global_intensity             0
Sub_metering_1               0
Sub_metering_2               0
Sub_metering_3           25979
dtype: int64
Duplicated Values:  0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   str    
 1   Time                   str    
 2   Global_active_power    object 
 3   Global_reactive_power  object 
 4   Voltage                object 
 5   Global_intensity       object 
 6   Sub_metering_1         object 
 7   Sub_metering_2         object 
 8   Sub_metering_3         float64
dtypes: float64(1), object(6), str(2)
memory usage: 176.0+ MB


In [5]:
df=df.replace("?",np.nan)

In [6]:
df.isnull().sum()

Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

In [7]:
numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

In [8]:
df[numeric_cols].dtypes

Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

In [9]:
df[numeric_cols] = df[numeric_cols].interpolate(method="linear")

In [10]:
df[numeric_cols] = df[numeric_cols].ffill().bfill()

In [11]:
df.isnull().sum()

Date                     0
Time                     0
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64

In [12]:
df["Datetime"]=pd.to_datetime(
    df["Date"]+ " "+ df["Time"],dayfirst=True
)

In [13]:
df[["Date", "Time", "Datetime"]].head()

,Date,Time,Datetime
0,16/12/2006,17:24:00,2006-12-16 17:24:00
1,16/12/2006,17:25:00,2006-12-16 17:25:00
2,16/12/2006,17:26:00,2006-12-16 17:26:00
3,16/12/2006,17:27:00,2006-12-16 17:27:00
4,16/12/2006,17:28:00,2006-12-16 17:28:00


In [14]:
df["Year"] = df["Datetime"].dt.year
df["Month"] = df["Datetime"].dt.month
df["Day"] = df["Datetime"].dt.day
df["DayOfWeek"] = df["Datetime"].dt.dayofweek
df["Hour"] = df["Datetime"].dt.hour
df["Minute"] = df["Datetime"].dt.minute

In [15]:
df[["Datetime", "Year", "Month", "Day", "DayOfWeek", "Hour", "Minute"]].head()

,Datetime,Year,Month,Day,DayOfWeek,Hour,Minute
0,2006-12-16 17:24:00,2006,12,16,5,17,24
1,2006-12-16 17:25:00,2006,12,16,5,17,25
2,2006-12-16 17:26:00,2006,12,16,5,17,26
3,2006-12-16 17:27:00,2006,12,16,5,17,27
4,2006-12-16 17:28:00,2006,12,16,5,17,28


In [16]:
df.drop(["Date", "Time", "Datetime"], axis=1, inplace=True)

In [17]:
df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Year,Month,Day,DayOfWeek,Hour,Minute
0,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006,12,16,5,17,24
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006,12,16,5,17,25
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006,12,16,5,17,26
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006,12,16,5,17,27
4,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006,12,16,5,17,28


In [18]:
df.dtypes

Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
Year                       int32
Month                      int32
Day                        int32
DayOfWeek                  int32
Hour                       int32
Minute                     int32
dtype: object

In [19]:
X=df.drop("Global_active_power",axis=1)
y=df["Global_active_power"]

In [20]:
print(X.shape)
print(y.shape)

(2075259, 12)
(2075259,)


In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [22]:
basic_model=LinearRegression()
basic_model.fit(X_train,y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [23]:
basic_model_pred=basic_model.predict(X_test)
mae=mean_absolute_error(y_test,basic_model_pred)
mse=mean_squared_error(y_test,basic_model_pred)
rmse=np.sqrt(mse)
r2=r2_score(y_test,basic_model_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R² Score:", r2)


MAE: 0.02570148512418765
MSE: 0.001636671733409684
RMSE: 0.04045579974997014
R² Score: 0.9985241861782174


In [24]:
df.corr(numeric_only=True)["Global_active_power"].sort_values(ascending=False)


Global_active_power      1.000000
Global_intensity         0.998887
Sub_metering_3           0.639272
Sub_metering_1           0.483816
Sub_metering_2           0.433892
Hour                     0.279776
Global_reactive_power    0.245047
DayOfWeek                0.061637
Minute                   0.002430
Day                     -0.000023
Year                    -0.032399
Month                   -0.033133
Voltage                 -0.395522
Name: Global_active_power, dtype: float64

In [25]:
X=df.drop("Global_active_power",axis=1)
y=df["Global_active_power"]

X=X.drop("Global_intensity",axis=1)

In [26]:
X_train = X.iloc[:int(len(X) * 0.8)]
X_test = X.iloc[int(len(X) * 0.8):]

y_train = y.iloc[:int(len(y) * 0.8)]
y_test = y.iloc[int(len(y) * 0.8):]

In [27]:
model=LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [28]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R² Score:", r2)

MAE: 0.2870269480361347
MSE: 0.14744543111298378
RMSE: 0.3839862381817658
R² Score: 0.8081707291599702


In [29]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [30]:
models={
    "Forest":Pipeline([
        ("model",RandomForestRegressor(random_state=42))
    ]),
    "tree":Pipeline([
        ("model",DecisionTreeRegressor(random_state=42))
    ]),
    "gradient_boosting":Pipeline([
        ("model",GradientBoostingRegressor(random_state=42))
    ])
}


In [31]:
for name,model in models.items():
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print("MAE:", mae)
    print("MSE:", mse)
    print("RMSE:", rmse)
    print("R² Score:", r2)
    

MAE: 0.23886724836099088
MSE: 0.14463482512654643
RMSE: 0.38030885491472116
R² Score: 0.8118273802540537
MAE: 0.29958764962478707
MSE: 0.28061086218920345
RMSE: 0.5297271582514941
R² Score: 0.6349200061526568
MAE: 0.2107546606834241
MSE: 0.10686608946229217
RMSE: 0.32690379236450007
R² Score: 0.86096521360931


In [33]:
import joblib

joblib.dump(models["gradient_boosting"],"electricity_consumtion_prediction_model.pkl")


['electricity_consumtion_prediction_model.pkl']